# WattBot 2025: RAG pipeline

Answers questions about AI's energy, water and carbon footprint from 32 documents, with citations.

1. Download the 32 PDFs listed in `metadata.csv` and cut them into page-level chunks
2. Hybrid retrieval: BM25 keywords + `bge-small` embeddings, fused with reciprocal rank fusion
3. `databricks-gpt-oss-120b` answers **only from the retrieved excerpts**, or marks the question unanswerable
4. Score on the 41 training questions with an offline copy of the WattBot metric, then write `submission.csv`

**Before running:** Settings → Internet **on**; Add-ons → Secrets: `DATABRICKS_HOST`, `DATABRICKS_TOKEN`;
Input → add the **WattBot 2025** competition data.

In [ ]:
!pip install -q pdfplumber
!rm -rf wattbot-rag && git clone -q https://github.com/BensonAnalyst/wattbot-rag.git
import sys; sys.path.insert(0, "wattbot-rag/src")

In [ ]:
import glob, json, os
import pandas as pd
from kaggle_secrets import UserSecretsClient

from wattbot.docs import build_chunks, download_pdfs, first_author, load_metadata
from wattbot.llm import make_databricks_llm
from wattbot.retrieve import Retriever, doc_recall
from wattbot.run import answer_all, make_embedder
from wattbot.score import parse_refs, score_frame

DATA = [d for d in glob.glob("/kaggle/input/*") if "wattbot" in d.lower()][0]
WORK = "/kaggle/working"
MODEL = "databricks-gpt-oss-120b"
K = 8  # excerpts per question

secrets = UserSecretsClient()
for name in ("DATABRICKS_HOST", "DATABRICKS_TOKEN"):
    os.environ[name] = secrets.get_secret(name)

meta = load_metadata(f"{DATA}/metadata.csv")
train = pd.read_csv(f"{DATA}/train_QA.csv")
test = pd.read_csv(f"{DATA}/test_Q.csv")
urls = dict(zip(meta["id"], meta["url"]))
authors = {r.id: first_author(r.citation) for r in meta.itertuples()}
print(f"{len(meta)} documents, {len(train)} train questions, {len(test)} test questions")

## 1. Documents

In [ ]:
pdfs = download_pdfs(meta, f"{WORK}/pdfs")
missing = [d for d, p in pdfs.items() if p is None]
print(f"Downloaded {len(pdfs) - len(missing)} of {len(pdfs)}. Missing: {missing or 'none'}")

chunks = build_chunks(meta, pdfs, cache_path=f"{WORK}/chunks.json")
per_doc = pd.Series([c["doc_id"] for c in chunks]).value_counts()
print(f"{len(chunks)} chunks; fewest: {per_doc.tail(3).to_dict()}")

## 2. Retrieval check on the training questions (no LLM): are the right documents retrieved?

In [ ]:
embed = make_embedder()
retrievers = {"bm25": Retriever(chunks, authors), "hybrid": Retriever(chunks, authors, embed_fn=embed)}

answerable = train[train["ref_id"] != "is_blank"]
for name, r in retrievers.items():
    recalls = [doc_recall(r.search(q.question, k=K), parse_refs(q.ref_id)) for q in answerable.itertuples()]
    print(f"{name:<7} document recall@{K}: {sum(recalls) / len(recalls):.3f}")

## 3. Answer the training questions and score them

In [ ]:
llm = make_databricks_llm(MODEL)
print("Model says:", llm([{"role": "user", "content": "Reply with the single word: ready"}]))

retriever = retrievers["hybrid"]
train_pred, train_ctx = answer_all(train, retriever, llm, urls, f"{WORK}/llm_cache.json", MODEL, k=K)
detail, train_score = score_frame(train_pred, train)
detail["type"] = [("true/false" if str(q).lower().startswith("true or false") else
                   "unanswerable" if t == "is_blank" else "text" if u == "is_blank" else "numeric")
                  for q, t, u in zip(train["question"], train["answer_value"], train["answer_unit"])]
print(f"Offline WattBot score on {len(train)} training questions: {train_score}")
display(detail.groupby("type")[["value", "ref", "na", "score"]].mean().round(3).assign(n=detail.groupby("type").size()))

In [ ]:
# Error analysis: every question that lost points
misses = detail[detail["score"] < 1].sort_values("score")
pd.set_option("display.max_colwidth", 120)
display(misses[["id", "type", "question", "truth", "pred", "truth_ref", "pred_ref", "score"]])
train_pred.merge(detail[["id", "score"]], on="id").to_csv(f"{WORK}/train_predictions.csv", index=False)

## 4. Answer the test questions and write the submission

In [ ]:
sub, test_ctx = answer_all(test, retriever, llm, urls, f"{WORK}/llm_cache.json", MODEL, k=K)
assert list(sub.columns) == list(train.columns) and len(sub) == len(test) and sub["id"].is_unique
sub.to_csv(f"{WORK}/submission.csv", index=False)
errors = sub["explanation"].astype(str).str.startswith("ERROR").sum()
print(f"submission.csv: {len(sub)} rows, {(sub['answer_value'] == 'is_blank').sum()} marked unanswerable, "
      f"{errors} failed calls")
display(sub.head(10))